<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/Q_and_A_Chatbot_and_RAG_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a ChatBot

# 🤖 Building a Q&A Chatbot with LangChain + RAG

This notebook builds a **Question-Answering chatbot** in two progressive stages:

**Stage 1 — Simple Chat Application**
A basic chatbot using LangChain's `ChatGroq` + `ChatPromptTemplate`.

**Stage 2 — RAG-Based Q&A (Retrieval-Augmented Generation)**
The chatbot is upgraded to answer questions from its *own custom knowledge base*
using vector embeddings and ChromaDB.

**Why does RAG matter?**
By default, an LLM only knows what it was trained on. RAG lets you plug in *your own documents*
so the model can answer questions about your data — without retraining.

---

## ⚙️ Setup — Install Dependencies

Run this cell if you haven't installed the required packages yet.
- `langchain-groq` — LangChain's integration with the Groq API
- `groq` — The official Groq Python client
- `python-dotenv` — Loads API keys from your `.env` file

In [ ]:
#! pip install langchain langchain-core langchain-groq groq

In [ ]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq
model = ChatGroq(
    temperature= 0.5,
    model='llama-3.1-8b-instant' # Groq Model
)



In [ ]:
response = model.invoke('What is Generative AI . Give answer in one sentence')
print(response.content)

Generative AI refers to a type of Artificial Intelligence that can create new, original content, such as images, music, text, or videos, using algorithms and machine learning models to generate novel and often surprising outputs.


---
# 🗨️ Stage 1: Building a Simple Chat Application

A **Prompt Template** defines the structure of every message sent to the model.
Instead of hard-coding prompts, templates let you reuse and customise prompts dynamically.

A LangChain chain connects components using the `|` (pipe) operator:
```
prompt_template | model
```
This means: format the input with the template, then send it to the model.

In [ ]:
from langchain_core.prompts import PromptTemplate

### 📝 Create a Prompt Template

We use `ChatPromptTemplate.from_messages()` to define a two-part prompt:
- **System message** — Sets the behaviour and persona of the AI (e.g., 'You are a helpful AI tutor')
- **Human message** — The actual user input, passed in dynamically via `{input}`

The system message is sent *once* to prime the model. Every user query fills the `{input}` placeholder.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Creating the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI Assistant. Respond to the user query clearly and concisely"),
    ("human","{input}")
])

### 🔗 Build the Chain and Run Interactive Chat

The chain `prompt | model` creates a pipeline:
1. User input is inserted into the prompt template
2. The formatted prompt is sent to the Groq model
3. The model's response is printed

The `while True` loop keeps the chat running until the user types `quit`.

> 💡 **Note for students:** This chatbot does *not* remember previous messages.
> Each question is answered independently. Memory will be covered in a later session.

In [ ]:
chain = prompt | model

# Interactive Chat

print("AI Tutor : Hello , I am your AI assist. Please feel free to ask me anything . And write quit to exit the conversation.")

while True :
  user_input = input("User : ")
  if user_input.lower() == 'quit':
    print("AI Tutor : Goodbye !")
    break
  response = chain.invoke({'input': user_input})
  print(f"AI Tutor : {response.content}")


AI Tutor : Hello , I am your AI assist. Please feel free to ask me anything . And write quit to exit the conversation.
User : quit
AI Tutor : Goodbye !


---
# 🔍 Stage 2: RAG-Based Q&A — Answering from a Knowledge Base

**What is RAG (Retrieval-Augmented Generation)?**

RAG is a technique that gives an LLM access to *external documents* at query time.
Instead of relying only on its training data, the model:
1. **Retrieves** the most relevant text chunks from your knowledge base
2. **Augments** the prompt with that context
3. **Generates** an answer grounded in your documents

**Components we'll build:**
- `ChromaDB` — A vector database that stores document embeddings
- `SentenceTransformerEmbeddings` — Converts text into numerical vectors
- `RecursiveCharacterTextSplitter` — Splits large documents into manageable chunks
- `Retriever` — Searches the vector DB for the most relevant chunks

In [ ]:
 #!pip install -q langchain langchain-community langchain-text-splitters chromadb sentence-transformers


In [ ]:
from langchain_community.vectorstores import Chroma # Vector Database
from langchain_community.embeddings import SentenceTransformerEmbeddings # Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter # Chunking


In [ ]:

# Sample documents (in a real project, load from files)
documents = [
    "The Apollo 11 mission landed humans on the Moon in 1969. Neil Armstrong and Buzz Aldrin were the first to walk on the lunar surface.",
    "SpaceX, founded by Elon Musk, launched the Falcon Heavy rocket in 2018, one of the most powerful rockets in history.",
    "The International Space Station (ISS) is a collaborative project between NASA, ESA, JAXA, CSA, and Roscosmos, orbiting Earth since 1998."
]

### ✂️ Split Documents into Chunks

`RecursiveCharacterTextSplitter` breaks documents into smaller overlapping pieces.

**Why chunk?**
- LLMs have a limited context window — you can't feed in an entire book at once
- Smaller chunks make retrieval more precise

**Parameters:**
- `chunk_size=500` — Each chunk is at most 500 characters
- `chunk_overlap=50` — Consecutive chunks share 50 characters to preserve context across boundaries

In [ ]:
# Splitting the documnts into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500 , chunk_overlap= 50)

chunks = text_splitter.create_documents(documents)


In [ ]:
chunks

[Document(metadata={}, page_content='The Apollo 11 mission landed humans on the Moon in 1969. Neil Armstrong and Buzz Aldrin were the first to walk on the lunar surface.'),
 Document(metadata={}, page_content='SpaceX, founded by Elon Musk, launched the Falcon Heavy rocket in 2018, one of the most powerful rockets in history.'),
 Document(metadata={}, page_content='The International Space Station (ISS) is a collaborative project between NASA, ESA, JAXA, CSA, and Roscosmos, orbiting Earth since 1998.')]

### 🧠 Create Embeddings and Store in ChromaDB

**What are embeddings?**
An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text.
Text with similar meaning will have vectors that are close together in mathematical space.

- `SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')` — A lightweight, fast embedding model
- `Chroma.from_documents()` — Embeds each chunk and stores them in an in-memory vector database

> 💡 **Analogy:** Think of embeddings as GPS coordinates — just as nearby places have similar coordinates,
> nearby topics have similar embedding vectors.

In [ ]:
# Create Embeddings

embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')

# String in Vector Database

vector_db = Chroma.from_documents(chunks, embedding_model)

/tmp/ipykernel_26754/2538943487.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 🔎 Create a Retriever

The retriever is the search engine of our RAG system.
Given a question, it finds the most semantically relevant chunks from ChromaDB.

- `search_kwargs={'k': 2}` — Retrieve the top **2** most relevant chunks

Increasing `k` gives more context but also more noise. `k=2` or `k=3` works well for small knowledge bases.

In [ ]:
# Create a retriver
retriver = vector_db.as_retriever(search_kwargs={'k': 2})

### 🧪 Test the Retriever

Let's test the retriever with two queries:
1. A question that *is* in our knowledge base (about the capital of France — which is NOT in our docs)

This is an important test — notice what happens when you ask about something *outside* the knowledge base.
The retriever will still return the 'closest' chunks it can find, which may not be relevant.

In [ ]:
relevant_chunks = retriver.invoke("What is the capital of France ?")
relevant_chunks

[Document(metadata={}, page_content='SpaceX, founded by Elon Musk, launched the Falcon Heavy rocket in 2018, one of the most powerful rockets in history.'),
 Document(metadata={}, page_content='The International Space Station (ISS) is a collaborative project between NASA, ESA, JAXA, CSA, and Roscosmos, orbiting Earth since 1998.')]

Combine the retrieved chunks into a single context string to pass into the prompt:

In [ ]:
context = " ".join(chunk.page_content for chunk in relevant_chunks)
context

'SpaceX, founded by Elon Musk, launched the Falcon Heavy rocket in 2018, one of the most powerful rockets in history. The International Space Station (ISS) is a collaborative project between NASA, ESA, JAXA, CSA, and Roscosmos, orbiting Earth since 1998.'

### 🤖 Build the Q&A Function

This function is the heart of our RAG pipeline:
1. **Retrieve** — Find the most relevant document chunks for the question
2. **Build context** — Join the chunks into a single string
3. **Prompt** — Ask the model to answer *based on the context only*
4. **Answer** — Return the model's grounded response

The key instruction `'If the context doesn't contain the answer, say so'` is critical —
it tells the model to *admit* when it doesn't know, rather than hallucinate an answer.

In [ ]:
# Function to answer questions
def answer_question(question):
    # Retrieve relevant chunks
    relevant_chunks = retriver.invoke(question)
    context = " ".join([chunk.page_content for chunk in relevant_chunks])

    # Create prompt
    prompt = f"""Context: {context}
Question: {question}
Answer based on the context provided. If the context doesn't contain the answer, say so."""

    # Generate response using Groq
    response = model.invoke(prompt)
    return response.content


In [ ]:
## Test the function

answer_question("What is the capital of France")

#answer_question("Who landed on the moon first?")

#answer_question("Where is ISS located?")

'The provided context does not mention the capital of France. It discusses SpaceX and the International Space Station, but does not include information about the capital of France.'